In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
import random
import numpy as np
import torch


def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cuda


In [3]:
from dataclasses import asdict, dataclass
from pathlib import Path
import json
import random

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.tensorboard import SummaryWriter


@dataclass
class Config:
    seed: int = 42
    batch_size: int = 32
    learning_rate: float = 1e-3
    epochs: int = 50
    patience: int = 8
    hidden_units: int = 64
    checkpoint_dir: str = "checkpoints/day54"
    log_dir: str = "runs/day54"


config = Config()

In [4]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(config.seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

dataset = load_breast_cancer()
X = dataset.data.astype(np.float32)
y = dataset.target.astype(np.float32)

# First separate the final test set.
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=config.seed,
    stratify=y,
)

# Split the remaining data into train and validation.
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val,
    y_train_val,
    test_size=0.20,
    random_state=config.seed,
    stratify=y_train_val,
)

# Fit preprocessing only on the training set.
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train).astype(np.float32)
X_val = scaler.transform(X_val).astype(np.float32)
X_test = scaler.transform(X_test).astype(np.float32)


def make_loader(X, y, batch_size, shuffle):
    dataset = TensorDataset(
        torch.tensor(X, dtype=torch.float32),
        torch.tensor(y, dtype=torch.float32).unsqueeze(1),
    )
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
    )


train_loader = make_loader(
    X_train, y_train, config.batch_size, shuffle=True
)
val_loader = make_loader(
    X_val, y_val, config.batch_size, shuffle=False
)
test_loader = make_loader(
    X_test, y_test, config.batch_size, shuffle=False
)

print(f"Device: {device}")
print(
    f"Train: {len(X_train)} | "
    f"Validation: {len(X_val)} | Test: {len(X_test)}"
)

Device: cuda
Train: 364 | Validation: 91 | Test: 114


In [5]:
class BinaryClassifier(nn.Module):
    def __init__(self, input_features: int, hidden_units: int):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_features, hidden_units),
            nn.ReLU(),
            nn.Dropout(p=0.2),
            nn.Linear(hidden_units, 1),
        )

    def forward(self, x):
        return self.network(x)


model = BinaryClassifier(
    input_features=X_train.shape[1],
    hidden_units=config.hidden_units,
).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=config.learning_rate,
    weight_decay=1e-4,
)

print(model)

BinaryClassifier(
  (network): Sequential(
    (0): Linear(in_features=30, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=64, out_features=1, bias=True)
  )
)


In [6]:
def run_epoch(model, loader, criterion, device, optimizer=None):
    training = optimizer is not None
    model.train(training)

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for features, labels in loader:
        features = features.to(device)
        labels = labels.to(device)

        if training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(training):
            logits = model(features)
            loss = criterion(logits, labels)

            if training:
                loss.backward()
                optimizer.step()

        probabilities = torch.sigmoid(logits)
        predictions = (probabilities >= 0.5).float()

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        total_correct += (predictions == labels).sum().item()
        total_samples += batch_size

    return {
        "loss": total_loss / total_samples,
        "accuracy": total_correct / total_samples,
    }

In [7]:
checkpoint_dir = Path(config.checkpoint_dir)
checkpoint_dir.mkdir(parents=True, exist_ok=True)

writer = SummaryWriter(log_dir=config.log_dir)

best_val_loss = float("inf")
epochs_without_improvement = 0
start_epoch = 0

# Set True when you want to resume from the latest checkpoint.
RESUME = False
latest_path = checkpoint_dir / "latest.pt"
best_path = checkpoint_dir / "best.pt"

if RESUME and latest_path.exists():
    checkpoint = torch.load(
        latest_path,
        map_location=device,
        weights_only=False,
    )

    model.load_state_dict(checkpoint["model_state"])
    optimizer.load_state_dict(checkpoint["optimizer_state"])
    start_epoch = checkpoint["epoch"] + 1
    best_val_loss = checkpoint["best_val_loss"]
    epochs_without_improvement = checkpoint["patience_counter"]

    print(f"Resuming from epoch {start_epoch + 1}")

for epoch in range(start_epoch, config.epochs):
    train_metrics = run_epoch(
        model, train_loader, criterion, device, optimizer
    )
    val_metrics = run_epoch(
        model, val_loader, criterion, device
    )

    # Log metrics so we can inspect learning curves later.
    for metric_name, metric_value in train_metrics.items():
        writer.add_scalar(
            f"train/{metric_name}", metric_value, epoch
        )

    for metric_name, metric_value in val_metrics.items():
        writer.add_scalar(
            f"validation/{metric_name}", metric_value, epoch
        )

    improved = val_metrics["loss"] < best_val_loss

    if improved:
        best_val_loss = val_metrics["loss"]
        epochs_without_improvement = 0

        torch.save(
            {
                "epoch": epoch,
                "model_state": model.state_dict(),
                "optimizer_state": optimizer.state_dict(),
                "best_val_loss": best_val_loss,
                "patience_counter": epochs_without_improvement,
                "config": asdict(config),
            },
            best_path,
        )
    else:
        epochs_without_improvement += 1

    # Save the latest state every epoch for crash recovery.
    torch.save(
        {
            "epoch": epoch,
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "best_val_loss": best_val_loss,
            "patience_counter": epochs_without_improvement,
            "config": asdict(config),
        },
        latest_path,
    )

    print(
        f"Epoch {epoch + 1:02d}/{config.epochs} | "
        f"train loss: {train_metrics['loss']:.4f} | "
        f"train acc: {train_metrics['accuracy']:.4f} | "
        f"val loss: {val_metrics['loss']:.4f} | "
        f"val acc: {val_metrics['accuracy']:.4f}"
    )

    if epochs_without_improvement >= config.patience:
        print("Early stopping: validation loss did not improve.")
        break

writer.close()
print(f"Best checkpoint: {best_path}")
print(f"Latest checkpoint: {latest_path}")

Epoch 01/50 | train loss: 0.6619 | train acc: 0.6374 | val loss: 0.5888 | val acc: 0.8681
Epoch 02/50 | train loss: 0.5333 | train acc: 0.9038 | val loss: 0.4814 | val acc: 0.9011
Epoch 03/50 | train loss: 0.4275 | train acc: 0.9231 | val loss: 0.3914 | val acc: 0.9231
Epoch 04/50 | train loss: 0.3448 | train acc: 0.9286 | val loss: 0.3157 | val acc: 0.9341
Epoch 05/50 | train loss: 0.2745 | train acc: 0.9286 | val loss: 0.2587 | val acc: 0.9341
Epoch 06/50 | train loss: 0.2294 | train acc: 0.9451 | val loss: 0.2163 | val acc: 0.9341
Epoch 07/50 | train loss: 0.1962 | train acc: 0.9505 | val loss: 0.1852 | val acc: 0.9341
Epoch 08/50 | train loss: 0.1692 | train acc: 0.9588 | val loss: 0.1631 | val acc: 0.9451
Epoch 09/50 | train loss: 0.1455 | train acc: 0.9615 | val loss: 0.1454 | val acc: 0.9451
Epoch 10/50 | train loss: 0.1282 | train acc: 0.9698 | val loss: 0.1320 | val acc: 0.9560
Epoch 11/50 | train loss: 0.1181 | train acc: 0.9698 | val loss: 0.1222 | val acc: 0.9670
Epoch 12/5

In [8]:
checkpoint = torch.load(
    best_path,
    map_location=device,
    weights_only=True,
)

model.load_state_dict(checkpoint["model_state"])

test_metrics = run_epoch(
    model, test_loader, criterion, device
)

print(f"Test loss: {test_metrics['loss']:.4f}")
print(f"Test accuracy: {test_metrics['accuracy']:.4f}")

Test loss: 0.0827
Test accuracy: 0.9649


In [9]:
%load_ext tensorboard
%tensorboard --logdir runs

<IPython.core.display.Javascript object>

In [10]:
tensorboard --logdir runs

Reusing TensorBoard on port 6006 (pid 245), started 0:00:00 ago. (Use '!kill 245' to kill it.)

<IPython.core.display.Javascript object>

In [11]:
import joblib

artifact_dir = Path("checkpoints/day54")
artifact_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(scaler, artifact_dir / "scaler.joblib")

metadata = {
    "feature_names": dataset.feature_names.tolist(),
    "target_names": dataset.target_names.tolist(),
    "seed": config.seed,
}

with open(artifact_dir / "metadata.json", "w", encoding="utf-8") as file:
    json.dump(metadata, file, indent=2)

print("Saved scaler and feature metadata.")

Saved scaler and feature metadata.
